# 01 — Exploratory Data Analysis

**Purpose:** Understand the raw price data before any modelling decisions are made.

**Important:** Nothing discovered here should be fed back into the pipeline as a hardcoded rule. Doing so would be data snooping — a form of lookahead bias applied at the research level rather than the code level.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from src.data.loader import get_prices
from src.data.universe import get_universe
from src.config import TRAIN_START

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
# Load a small subset for exploration
universe = get_universe(use_full_sp500=False)
print(f"Universe: {len(universe)} tickers")

prices = get_prices(universe[:10], start=TRAIN_START)
print(f"\nLoaded {len(prices):,} rows")
print(f"Date range: {prices['date'].min().date()} to {prices['date'].max().date()}")
prices.head()

In [ ]:
# Check for missing data
missing = prices.groupby('ticker').apply(lambda g: g['close'].isna().sum())
print("Missing close values per ticker:")
print(missing[missing > 0] if missing.any() else 'None — clean data')

In [ ]:
# Normalised price chart
close_wide = prices.pivot_table(index='date', columns='ticker', values='close')
norm = close_wide.divide(close_wide.iloc[0]) * 100

fig = px.line(norm, title='Normalised Prices (rebased to 100 at start)')
fig.update_layout(yaxis_title='Rebased Price', xaxis_title='Date', height=450)
fig.show()

In [ ]:
# Distribution of daily returns
log_rets = close_wide.pct_change().stack().reset_index()
log_rets.columns = ['date', 'ticker', 'return']

fig = px.histogram(
    log_rets, x='return', facet_col='ticker', facet_col_wrap=3,
    nbins=80, title='Distribution of Daily Returns by Ticker'
)
fig.update_xaxes(range=[-0.12, 0.12])
fig.show()

In [ ]:
# Rolling 20-day volatility
vol = close_wide.pct_change().rolling(20).std() * np.sqrt(252)

fig = px.line(vol, title='Annualised Rolling 20-Day Volatility')
fig.update_layout(yaxis_title='Annualised Vol', yaxis_tickformat='.0%', height=400)
fig.show()

In [ ]:
# Correlation matrix of daily returns
import matplotlib.pyplot as plt

ret_wide = close_wide.pct_change().dropna()
corr = ret_wide.corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im)
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
ax.set_title('Return Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("Summary statistics for daily returns:")
ret_wide.describe().applymap('{:.4f}'.format)